# Evaluating the final strategy over 2025

The final strategy is the Avellaneda–Stoikov market maker with the LightGBM adverse-selection premium, the queue-aware $k$, γ = 0.5 and a green-up threshold of 1 contract. It was backtested on every WIN race of 2025, twice: once as it is, and once with **diming** (`dime=True`), where a quote inside the spread goes at most one tick in front of the best price on its side.

**What was fitted where.** It matters which days the strategy has already seen:

| Period | Used for |
|---|---|
| January | estimating σ², $k$ and the markout horizon, and training the adverse-selection model |
| 1–14 February | tuning γ and the green-up threshold |
| **15 February – 31 December** | **nothing: this is the out-of-sample test** |

Results on January and early February flatter the strategy, so every headline number below is for the out-of-sample period, with the earlier part shown separately for reference.

Each backtest uses the same engine settings: 0.2 s latency, the queue rules in `backtest/execution.py`, and 4% commission on each market's net winnings.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Project root = the folder that holds research/ (works whether this runs from evaluation/ or the root)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research" / "results").exists())
RUNS = {"no diming": ROOT / "research/results/final_2025_no_dime",
        "diming": ROOT / "research/results/final_2025_dime"}
OUT_OF_SAMPLE_FROM = pd.Timestamp("2025-02-15").date()      # the first day nothing was fitted or tuned on
N_BOOT = 1000
rng = np.random.default_rng(42)

# Chart style: blue = no diming, orange = diming, greys for everything else
COLOURS = {"no diming": "#2a78d6", "diming": "#eb6834"}
GREY, LIGHT = "#898781", "#c3c2b7"
INK, INK_2 = "#0b0b0b", "#52514e"
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({
    "figure.figsize": (8, 4.2), "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.size": 10, "text.color": INK, "axes.labelcolor": INK_2, "axes.labelpad": 6,
    "xtick.color": LIGHT, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2, "ytick.major.size": 0,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.edgecolor": LIGHT, "axes.linewidth": 0.8,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "lines.linewidth": 2, "lines.markersize": 7, "lines.markeredgewidth": 1.5, "lines.markeredgecolor": "white",
    "legend.frameon": False, "legend.fontsize": 9.5,
})


def titles(ax, title, subtitle, gap=10):
    # A bold headline that states the finding, then a smaller line saying what is plotted
    note = ax.annotate(subtitle, (0, 1), xycoords="axes fraction", xytext=(0, gap), textcoords="offset points",
                       fontsize=9.5, color=INK_2, va="bottom", linespacing=1.45)
    head = ax.set_title(title, loc="left", fontsize=12.5, fontweight="semibold",
                        pad=gap + 6 + 14 * (subtitle.count("\n") + 1))
    note.set_in_layout(False)
    head.set_in_layout(False)


def legend(ax, **kwargs):
    # A legend whose keys drop the white ring around markers (on a short key it makes a line look dashed)
    for key in ax.legend(**kwargs).legend_handles:
        if isinstance(key, Line2D):
            key.set_markeredgewidth(0)


missing = [name for name, folder in RUNS.items() if not (folder / "markets.csv").exists()]
RUNS = {name: folder for name, folder in RUNS.items() if name not in missing}
print("Runs found:", ", ".join(RUNS) or "none", "| still missing:", ", ".join(missing) or "none")

## Step 1: Loading the runs

From each backtest we take every race's profit after commission (`markets.csv`) and every fill (`fills.csv`). A race is dated by its first fill, on the Australian racing day it was run. Every race traded by either version goes into one table, with zero for a version that didn't trade it, so the two are compared on the same races.

In [ ]:
def load_run(folder):
    markets = pd.read_csv(folder / "markets.csv", dtype={"market_id": str}).set_index("market_id")
    fills = pd.read_csv(folder / "fills.csv", dtype={"market_id": str})
    fills["time"] = pd.to_datetime(fills["publish_time"], format="ISO8601", utc=True)
    return markets, fills


runs = {name: load_run(folder) for name, folder in RUNS.items()}

first_fill = pd.concat([fills.groupby("market_id")["time"].min() for _, fills in runs.values()]).groupby(level=0).min()
race_time = first_fill.sort_values()
RACES = race_time.index                                         # in the order they were run
day = race_time.dt.tz_convert("Australia/Sydney").dt.date
month = race_time.dt.tz_convert("Australia/Sydney").dt.strftime("%b")
out_of_sample = (day >= OUT_OF_SAMPLE_FROM).to_numpy()
profit = pd.DataFrame({name: markets["net_pnl"].reindex(RACES, fill_value=0.0) for name, (markets, _) in runs.items()})
print(f"{len(RACES):,} races traded: {(~out_of_sample).sum():,} before 15 February, {out_of_sample.sum():,} from then on")

## Step 2: The headline numbers

For each version and period:

- **profit** after commission, with a 95% interval from resampling races (as in the earlier notebooks);
- **profit per race traded**, and the share of racing days that made money;
- **daily Sharpe ratio**: average daily profit divided by its standard deviation, and annualised (× √365, since there is racing every day);
- **largest drawdown**: the biggest fall in cumulative profit from a previous high;
- **fills**: how many of our quotes were filled.

In [ ]:
def interval(values):
    ordered = np.sort(np.asarray(values))
    return ordered[int(0.025 * len(ordered))], ordered[int(0.975 * len(ordered)) - 1]


def headline(name, chosen):
    races = profit.loc[chosen, name]
    daily = races.groupby(day[chosen]).sum()
    cumulative = races.cumsum()
    draws = rng.multinomial(len(races), np.full(len(races), 1 / len(races)), size=N_BOOT)
    low, high = interval(draws @ races.to_numpy())
    markets, fills = runs[name]
    in_period = fills["market_id"].isin(races.index)
    return {"profit ($)": races.sum(), "95% CI low": low, "95% CI high": high,
            "races traded": int(markets.index.isin(races.index).sum()),
            "profit per race traded ($)": races.sum() / max(markets.index.isin(races.index).sum(), 1),
            "profitable days": f"{(daily > 0).sum()}/{len(daily)}",
            "daily Sharpe": daily.mean() / daily.std(), "annualised Sharpe": daily.mean() / daily.std() * np.sqrt(365),
            "largest drawdown ($)": (cumulative.cummax() - cumulative).max(),
            "quote fills": int(((fills["tag"] == "quote") & in_period).sum())}


PERIODS = {"out of sample (15 Feb – 31 Dec)": out_of_sample, "fitted and tuned on (Jan – 14 Feb)": ~out_of_sample}
summary = pd.DataFrame({(period, name): headline(name, chosen) for period, chosen in PERIODS.items() for name in RUNS}).T
summary.round(2)

## Step 3: Profit through the year

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
start = race_time.iloc[np.argmax(out_of_sample)] if out_of_sample.any() else race_time.iloc[-1]
ax.axvspan(race_time.iloc[0], start, color="#f4f3ef", lw=0, zorder=0)
ax.text(race_time.iloc[0], 0.97, "  fitted and tuned on", transform=ax.get_xaxis_transform(), va="top", fontsize=9,
        color=INK_2)
for name in RUNS:
    ax.plot(race_time.to_numpy(), profit[name].cumsum().to_numpy(), color=COLOURS[name], label=name.capitalize())
ax.axhline(0, color=LIGHT, lw=1)
ax.set(ylabel="Cumulative profit after commission ($)")
legend(ax, loc="upper left", bbox_to_anchor=(0, 0.9))
titles(ax, "Cumulative profit through 2025",
       "Every race in the order it was run. The shaded start of the year is what the strategy was fitted and tuned on.")
plt.show()

In [ ]:
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly = profit.groupby(month.rename("month")).sum().reindex([m for m in MONTHS if m in set(month)])

fig, ax = plt.subplots(figsize=(11, 4.2))
width = 0.8 / len(RUNS)
for i, name in enumerate(RUNS):
    ax.bar(np.arange(len(monthly)) + (i - (len(RUNS) - 1) / 2) * width, monthly[name], width=width * 0.9,
           color=COLOURS[name], label=name.capitalize())
ax.axhline(0, color=INK_2, lw=1)
ax.set_xticks(np.arange(len(monthly)), monthly.index)
ax.set(ylabel="Profit after commission ($)")
legend(ax, loc="upper left")
titles(ax, "Profit month by month", "Profit after commission in each month of 2025 (January and early February were "
       "used for fitting and tuning)")
plt.show()

monthly.round(2)

## Step 4: Does diming help?

Diming changes where some quotes sit, so the two versions trade many of the same races. The fair comparison is race by race: for each race, profit with diming minus profit without, resampled by race for a paired 95% interval.

In [ ]:
if len(RUNS) == 2:
    difference = (profit["diming"] - profit["no diming"])[out_of_sample]
    draws = rng.multinomial(len(difference), np.full(len(difference), 1 / len(difference)), size=N_BOOT)
    low, high = interval(draws @ difference.to_numpy())
    daily_difference = difference.groupby(day[out_of_sample]).sum()
    print(f"Out of sample, diming changes profit by ${difference.sum():+.2f} (95% CI {low:+.2f} to {high:+.2f}); "
          f"it's ahead on {(daily_difference > 0).sum()} of {len(daily_difference)} days")
else:
    print("Needs both runs")

## Still to do, once the full-year runs are in

- **Where the profit comes from.** Split each fill's profit into the spread we captured (the quote's distance from the mid when posted), the adverse move after it (the markout), the cost of greening up at $T$, and what was left to settle on the result.
- **When and where it works.** Profit by time to the off, by the runner's odds, and by distance from the mid, to see whether the edge comes from a few situations or is spread evenly.
- **Inventory.** How large positions get and how often the cap of 50 contracts binds.
- **Conclusions** for the report: is the out-of-sample profit distinguishable from zero, and should the final strategy use diming?